<a href="https://colab.research.google.com/github/fariasad/UCSC-Ciencia-de-Datos/blob/main/S2_Laboratorio_ARIAS_FREDDY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | Freddy Antonio Arias Adasme |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [3]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = 'Online Retail.xlsx'

df = pd.read_excel(
    RUTA + ARCHIVO,
    engine='openpyxl'
)

print('Forma:', df.shape)
display(df.head())

Forma: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.dtypes

,0
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,float64
Country,object


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [5]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [6]:
# Actividad 1.1
# Analizar valores faltantes de CustomerID por país

faltantes_pais = (
    df.groupby('Country')
      .agg(
          total_registros=('CustomerID', 'size'),
          sin_customerid=('CustomerID', lambda x: x.isna().sum())
      )
)

faltantes_pais['porcentaje_faltantes'] = (
    faltantes_pais['sin_customerid']
    / faltantes_pais['total_registros'] * 100
).round(2)

# Mostrar países con mayor cantidad de faltantes
display(
    faltantes_pais.sort_values(
        'sin_customerid',
        ascending=False
    ).head(15)
)

print(
    "Porcentaje global de faltantes:",
    round(df['CustomerID'].isna().mean() * 100, 2),
    "%"
)

,total_registros,sin_customerid,porcentaje_faltantes
Country,,,
United Kingdom,495478,133600,26.96
EIRE,8196,711,8.67
Hong Kong,288,288,100.00
Unspecified,446,202,45.29
Switzerland,2002,125,6.24
France,8557,66,0.77
Israel,297,47,15.82
Portugal,1519,39,2.57
Bahrain,19,2,10.53


Porcentaje global de faltantes: 24.93 %


### 1.2 Exactitud — rangos y valores imposibles

In [7]:
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
Quantity,541909.0,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [8]:
# Actividad 1.2 — Revisar valores extremos

for columna in ['Quantity', 'UnitPrice']:
    print(f"\n=== {columna} ===")

    print("Mínimo:", df[columna].min())
    print("Máximo:", df[columna].max())

    print("\nValores más bajos:")
    print(df[columna].value_counts().sort_index().head(10))

    print("\nValores más altos:")
    print(df[columna].value_counts().sort_index().tail(10))



=== Quantity ===
Mínimo: -80995
Máximo: 80995

Valores más bajos:
Quantity
-80995    1
-74215    1
-9600     2
-9360     1
-9058     1
-5368     1
-4830     1
-3667     1
-3167     1
-3114     1
Name: count, dtype: int64

Valores más altos:
Quantity
3114     2
3186     1
3906     1
4000     1
4300     1
4800     1
5568     1
12540    1
74215    1
80995    1
Name: count, dtype: int64

=== UnitPrice ===
Mínimo: -11062.06
Máximo: 38970.0

Valores más bajos:
UnitPrice
-11062.060       2
 0.000        2515
 0.001           4
 0.010           1
 0.030           3
 0.040          66
 0.060         117
 0.070           9
 0.080          56
 0.090           2
Name: count, dtype: int64

Valores más altos:
UnitPrice
8142.75     2
8286.22     1
11062.06    1
11586.50    1
13474.79    1
13541.33    3
16453.71    1
16888.02    1
17836.46    1
38970.00    1
Name: count, dtype: int64


In [9]:
# Revisar registros con cantidades y precios extremos

sospechosos = df.loc[
    (df['Quantity'].abs() >= 10000) |
    (df['UnitPrice'] < 0)
]

display(
    sospechosos[
        ['InvoiceNo', 'StockCode', 'Description',
         'Quantity', 'UnitPrice', 'InvoiceDate']
    ].head(30)
)

print("Registros encontrados:", len(sospechosos))

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00
299983,A563186,B,Adjust bad debt,1,-11062.06,2011-08-12 14:51:00
299984,A563187,B,Adjust bad debt,1,-11062.06,2011-08-12 14:52:00
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,2011-11-25 15:57:00
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00


Registros encontrados: 7


### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [10]:
# Regla 1: Las facturas identificadas como cancelaciones mediante el prefijo C deberían tener cantidades negativas.

# Regla 2: Las transacciones comerciales ordinarias deberían tener precios unitarios no negativos; los precios negativos deberán identificarse como posibles ajustes.

# Actividad 1.3 — Reglas de consistencia

# Regla 1: Cancelaciones con cantidad negativa
cancelacion = df['InvoiceNo'].astype(str).str.startswith('C')

regla1 = df.loc[
    cancelacion & (df['Quantity'] >= 0)
]

print("Regla 1:")
print("Cancelaciones con cantidad no negativa:", len(regla1))

# Regla 2: Precios negativos
regla2 = df.loc[df['UnitPrice'] < 0]

print("\nRegla 2:")
print("Registros con precio negativo:", len(regla2))

display(
    regla2[['InvoiceNo', 'StockCode',
            'Description', 'Quantity', 'UnitPrice']]
)


Regla 1:
Cancelaciones con cantidad no negativa: 0

Regla 2:
Registros con precio negativo: 2


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
299983,A563186,B,Adjust bad debt,1,-11062.06
299984,A563187,B,Adjust bad debt,1,-11062.06


### 1.4 Unicidad

In [11]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

Duplicados exactos: 5268


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [12]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Actividad 1.4 — Clave normalizada y casi-duplicados

# Construir una clave con los campos de la transacción
clave_normalizada = pd.DataFrame({
    'InvoiceNo': normalizar_texto(df['InvoiceNo']),
    'StockCode': normalizar_texto(df['StockCode']),
    'Description': normalizar_texto(df['Description']),
    'Quantity': df['Quantity'],
    'InvoiceDate': df['InvoiceDate'],
    'UnitPrice': df['UnitPrice'],
    'CustomerID': df['CustomerID'],
    'Country': normalizar_texto(df['Country'])
})

# Comparar duplicados exactos y normalizados
duplicados_exactos = df.duplicated().sum()
duplicados_normalizados = clave_normalizada.duplicated().sum()

print("Duplicados exactos:", duplicados_exactos)
print("Duplicados con clave normalizada:", duplicados_normalizados)
print(
    "Casi-duplicados adicionales:",
    duplicados_normalizados - duplicados_exactos
)


Duplicados exactos: 5268
Duplicados con clave normalizada: 5268
Casi-duplicados adicionales: 0


### 1.5 Oportunidad

In [14]:
# Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?
# ¿El conjunto está vigente para la pregunta que quieres responder?

print("Fecha inicial:", df['InvoiceDate'].min())
print("Fecha más reciente:", df['InvoiceDate'].max())

Fecha inicial: 2010-12-01 08:26:00
Fecha más reciente: 2011-12-09 12:50:00


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [15]:
# Trabajar sobre una copia del dataset original
df_limpio = df.copy()

# Buscar posibles valores centinela en campos de texto
centinelas_texto = [
    'S/I', 'SIN INFORMACION', 'N/A',
    'NULL', 'NONE', 'DESCONOCIDO'
]

columnas_texto = [
    'InvoiceNo', 'StockCode', 'Description', 'Country'
]

print("=== POSIBLES CENTINELAS DE TEXTO ===")

for columna in columnas_texto:
    valores = df_limpio[columna].astype('string').str.strip().str.upper()
    encontrados = valores.isin(centinelas_texto).sum()
    print(f"{columna}: {encontrados}")

print("\n=== VALORES NUMÉRICOS A REVISAR ===")

for columna in ['Quantity', 'UnitPrice']:
    print(f"\n{columna}")
    print("Valores -999:", (df_limpio[columna] == -999).sum())
    print("Valores 9999:", (df_limpio[columna] == 9999).sum())
    print("Valores -1:", (df_limpio[columna] == -1).sum())
    print("Valores 0:", (df_limpio[columna] == 0).sum())


=== POSIBLES CENTINELAS DE TEXTO ===
InvoiceNo: 0
StockCode: 0
Description: 0
Country: 0

=== VALORES NUMÉRICOS A REVISAR ===

Quantity
Valores -999: 0
Valores 9999: 0
Valores -1: 4184
Valores 0: 0

UnitPrice
Valores -999: 0
Valores 9999: 0
Valores -1: 0
Valores 0: 2515


Al revisar posibles valores centinela no encontré coincidencias en las columnas de texto. En los campos numéricos aparecieron 4.184 registros con Quantity = -1 y 2.515 con UnitPrice = 0.

Decidí mantener estos valores por ahora, ya que las cantidades negativas pueden corresponder a devoluciones y los precios cero podrían representar operaciones especiales. No sería correcto reemplazarlos por valores faltantes sin comprobar su significado.

### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [18]:
# Actividad 2.2 — Normalización de categorías

# Trabajar sobre la copia creada en la Actividad 2.1
# No modificar el DataFrame original df

def normalizar_categoria(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', 'ignore')
        .str.decode('ascii')
    )

# Revisar categorías originales y normalizadas
pais_original = df_limpio['Country']
pais_normalizado = normalizar_categoria(pais_original)

print("=== COUNTRY ===")
print("Categorías originales:", pais_original.nunique())
print("Categorías normalizadas:", pais_normalizado.nunique())

print("\nCategorías después de normalizar:")
print(sorted(pais_normalizado.dropna().unique()))

# Identificar valores que cambian con la normalización
cambios = df_limpio.loc[
    pais_original.astype('string') != pais_normalizado,
    ['Country']
].copy()

print("\nRegistros con cambios de escritura:", len(cambios))

# Revisar posibles nombres equivalentes
print("\nCategorías relacionadas con Irlanda:")
print([
    pais for pais in pais_normalizado.dropna().unique()
    if 'IRE' in pais
])

print("\nFrecuencia de EIRE:")
print((pais_normalizado == 'EIRE').sum())

# Diccionario de equivalencias
EQUIVALENCIAS_PAIS = {
    'EIRE': 'IRELAND'
}

# Crear columna normalizada sin modificar la original
df_limpio['Country_norm'] = pais_normalizado.replace(
    EQUIVALENCIAS_PAIS
)

# Verificar resultados
print("Categorías finales:", df_limpio['Country_norm'].nunique())

print(
    "Registros normalizados de EIRE a IRELAND:",
    (df_limpio['Country_norm'] == 'IRELAND').sum()
)

display(
    df_limpio[['Country', 'Country_norm']]
    .drop_duplicates()
    .head(15)
)

=== COUNTRY ===
Categorías originales: 38
Categorías normalizadas: 38

Categorías después de normalizar:
['AUSTRALIA', 'AUSTRIA', 'BAHRAIN', 'BELGIUM', 'BRAZIL', 'CANADA', 'CHANNEL ISLANDS', 'CYPRUS', 'CZECH REPUBLIC', 'DENMARK', 'EIRE', 'EUROPEAN COMMUNITY', 'FINLAND', 'FRANCE', 'GERMANY', 'GREECE', 'HONG KONG', 'ICELAND', 'ISRAEL', 'ITALY', 'JAPAN', 'LEBANON', 'LITHUANIA', 'MALTA', 'NETHERLANDS', 'NORWAY', 'POLAND', 'PORTUGAL', 'RSA', 'SAUDI ARABIA', 'SINGAPORE', 'SPAIN', 'SWEDEN', 'SWITZERLAND', 'UNITED ARAB EMIRATES', 'UNITED KINGDOM', 'UNSPECIFIED', 'USA']

Registros con cambios de escritura: 533364

Categorías relacionadas con Irlanda:
['EIRE']

Frecuencia de EIRE:
8196
Categorías finales: 38
Registros normalizados de EIRE a IRELAND: 8196


,Country,Country_norm
0,United Kingdom,UNITED KINGDOM
26,France,FRANCE
197,Australia,AUSTRALIA
385,Netherlands,NETHERLANDS
1109,Germany,GERMANY
1236,Norway,NORWAY
1404,EIRE,IRELAND
5320,Switzerland,SWITZERLAND
6421,Spain,SPAIN
6608,Poland,POLAND


### ✏️ Actividad 2.3 — Duplicados

In [20]:
# Actividad 2.3 — Revisión de duplicados

# Identificar todas las filas pertenecientes a grupos duplicados
duplicados = df.loc[
    df.duplicated(keep=False)
].copy()

# Contar duplicados exactos
print("Duplicados exactos:", df.duplicated().sum())

# Mostrar ejemplos para revisar
columnas = [
    'InvoiceNo',
    'StockCode',
    'Description',
    'Quantity',
    'UnitPrice',
    'CustomerID'
]

display(
    duplicados[columnas]
    .sort_values(['InvoiceNo', 'StockCode'])
    .head(20)
)

# Evaluar cuántas facturas están involucradas
print(
    "Facturas con registros duplicados:",
    duplicados['InvoiceNo'].nunique()
)

# Impacto potencial de los duplicados en los ingresos

duplicados_extra = df.loc[df.duplicated(keep='first')].copy()

duplicados_extra['importe'] = (
    duplicados_extra['Quantity'] *
    duplicados_extra['UnitPrice']
)

print("Duplicados exactos:", len(duplicados_extra))
print(
    "Importe neto asociado a filas duplicadas adicionales:",
    round(duplicados_extra['importe'].sum(), 2)
)

Duplicados exactos: 5268


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,1.25,17908.0
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,1.25,17908.0
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,4.95,17908.0
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,4.95,17908.0
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2.10,17908.0
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2.10,17908.0
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2.95,17908.0
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2.95,17908.0
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,1.65,17920.0
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,1.65,17920.0


Facturas con registros duplicados: 1933
Duplicados exactos: 5268
Importe neto asociado a filas duplicadas adicionales: 21740.98


Se encontraron 5.268 registros duplicados exactos, distribuidos en 1.933 facturas. Al revisar algunos ejemplos, observé que una misma factura puede contener varias líneas del mismo producto.

Por ahora decidí conservar estos registros, ya que no existe un identificador único por línea que permita confirmar cuáles están duplicados por error. Eliminarlos sin comprobarlo podría afectar el cálculo de ingresos. Esta situación quedará registrada como una limitación del conjunto de datos.

También calculé que las filas duplicadas adicionales representan un importe neto de 21.740,98. Este valor permite estimar el impacto que tendría eliminarlas, pero no significa que todas sean errores. Por eso decidí mantenerlas y dejar registrada esta limitación.

### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [21]:
# Actividad 2.4 — Conversión de tipos de dato

print("=== TIPOS ANTES ===")
print(df_limpio.dtypes)

# Convertir cantidades y precios a numéricos
df_limpio['Quantity'] = pd.to_numeric(
    df_limpio['Quantity'], errors='coerce'
)

df_limpio['UnitPrice'] = pd.to_numeric(
    df_limpio['UnitPrice'], errors='coerce'
)

# Convertir fecha con formato explícito
df_limpio['InvoiceDate'] = pd.to_datetime(
    df_limpio['InvoiceDate'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# CustomerID es un identificador, no una cantidad
# Primero convertir a entero nullable y luego a texto
df_limpio['CustomerID'] = (
    pd.to_numeric(df_limpio['CustomerID'], errors='coerce')
    .astype('Int64')
    .astype('string')
)

# Garantizar que los códigos sean texto
df_limpio['InvoiceNo'] = df_limpio['InvoiceNo'].astype('string')
df_limpio['StockCode'] = df_limpio['StockCode'].astype('string')

print("\n=== TIPOS DESPUÉS ===")
print(df_limpio.dtypes)

print("\n=== COMPROBACIONES ===")
print("Fechas inválidas:", df_limpio['InvoiceDate'].isna().sum())
print("Cantidades faltantes:", df_limpio['Quantity'].isna().sum())
print("Precios faltantes:", df_limpio['UnitPrice'].isna().sum())
print("CustomerID faltantes:", df_limpio['CustomerID'].isna().sum())

display(df_limpio.head())

=== TIPOS ANTES ===
InvoiceNo               object
StockCode               object
Description             object
Quantity                 int64
InvoiceDate     datetime64[ns]
UnitPrice              float64
CustomerID             float64
Country                 object
Country_norm            object
dtype: object

=== TIPOS DESPUÉS ===
InvoiceNo       string[python]
StockCode       string[python]
Description             object
Quantity                 int64
InvoiceDate     datetime64[ns]
UnitPrice              float64
CustomerID      string[python]
Country                 object
Country_norm            object
dtype: object

=== COMPROBACIONES ===
Fechas inválidas: 0
Cantidades faltantes: 0
Precios faltantes: 0
CustomerID faltantes: 135080


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Country_norm
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,UNITED KINGDOM
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,UNITED KINGDOM
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [22]:
# Actividad 2.5 — Análisis de valores atípicos

# Detectar atípicos mediante IQR
for columna in ['Quantity', 'UnitPrice']:

    q1 = df_limpio[columna].quantile(0.25)
    q3 = df_limpio[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    atipicos = df_limpio.loc[
        (df_limpio[columna] < limite_inferior) |
        (df_limpio[columna] > limite_superior)
    ]

    print(f"\n=== {columna} ===")
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Registros atípicos:", len(atipicos))

    display(
        atipicos[
            ['InvoiceNo', 'StockCode', 'Description',
             'Quantity', 'UnitPrice']
        ].head(10)
    )


=== Quantity ===
Límite inferior: -12.5
Límite superior: 23.5
Registros atípicos: 58619


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,1.69
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,3.75
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,3.75
30,536370,21883,STARS GIFT TAPE,24,0.65
31,536370,10002,INFLATABLE POLITICAL GLOBE,48,0.85
32,536370,21791,VINTAGE HEADS AND TAILS CARD GAME,24,1.25
34,536370,22326,ROUND SNACK BOXES SET OF4 WOODLAND,24,2.95
35,536370,22629,SPACEBOY LUNCH BOX,24,1.95
36,536370,22659,LUNCH BOX I LOVE LONDON,24,1.95
37,536370,22631,CIRCUS PARADE LUNCH BOX,24,1.95



=== UnitPrice ===
Límite inferior: -3.07
Límite superior: 8.45
Registros atípicos: 39627


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
16,536367,22622,BOX OF VINTAGE ALPHABET BLOCKS,2,9.95
45,536370,POST,POSTAGE,3,18.00
65,536374,21258,VICTORIAN SEWING BOX LARGE,32,10.95
141,C536379,D,Discount,-1,27.50
151,536382,22839,3 TIER CAKE TIN GREEN AND CREAM,2,14.95
152,536382,22838,3 TIER CAKE TIN RED AND CREAM,2,14.95
153,536382,22783,SET 3 WICKER OVAL BASKETS W LIDS,4,16.95
163,536384,21340,CLASSIC METAL BIRDCAGE PLANT HOLDER,2,12.75
167,536384,22424,ENAMEL BREAD BIN CREAM,8,10.95
168,536385,22783,SET 3 WICKER OVAL BASKETS W LIDS,1,19.95


In [23]:
# Actividad 2.5 — Clasificación de operaciones especiales

cancelaciones = (
    df_limpio['InvoiceNo']
    .astype('string')
    .str.startswith('C', na=False)
)

ajustes = (
    df_limpio['Description']
    .astype('string')
    .str.contains('ADJUST', case=False, na=False)
)

print("=== CLASIFICACIÓN DE REGISTROS ===")

print("Facturas de cancelación:", cancelaciones.sum())

print(
    "Cantidades negativas sin prefijo C:",
    ((df_limpio['Quantity'] < 0) & ~cancelaciones).sum()
)

print("Registros con descripción ADJUST:", ajustes.sum())

print(
    "Precios negativos:",
    (df_limpio['UnitPrice'] < 0).sum()
)

print(
    "Precios iguales a cero:",
    (df_limpio['UnitPrice'] == 0).sum()
)

print("\n=== OPERACIONES EXTREMAS ===")

display(
    df_limpio.loc[
        (df_limpio['Quantity'].abs() >= 10000) |
        (df_limpio['UnitPrice'] < 0),
        ['InvoiceNo', 'Description', 'Quantity',
         'UnitPrice', 'InvoiceDate']
    ]
)

=== CLASIFICACIÓN DE REGISTROS ===
Facturas de cancelación: 9288
Cantidades negativas sin prefijo C: 1336
Registros con descripción ADJUST: 34
Precios negativos: 2
Precios iguales a cero: 2515

=== OPERACIONES EXTREMAS ===


,InvoiceNo,Description,Quantity,UnitPrice,InvoiceDate
61619,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00
61624,C541433,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00
299983,A563186,Adjust bad debt,1,-11062.06,2011-08-12 14:51:00
299984,A563187,Adjust bad debt,1,-11062.06,2011-08-12 14:52:00
502122,578841,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,2011-11-25 15:57:00
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00
540422,C581484,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00


_Tu conclusión sobre los atípicos:_

Al revisar los valores atípicos encontré cantidades y precios que se alejan bastante de los valores habituales. Sin embargo, al revisar las facturas y descripciones, observé que varios corresponden a ventas de grandes cantidades, cancelaciones y ajustes contables.

Por eso decidí no eliminar los valores extremos automáticamente, ya que algunos representan operaciones reales. Los registros con precios cero y cantidades negativas sin factura de cancelación necesitan una revisión adicional. Los ajustes contables también deberán tratarse por separado para no afectar el análisis de ingresos por producto.



---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [24]:
# Actividad 3.1 — Crear un atributo derivado

# Calcular el importe de cada línea de factura
df_limpio['ImporteLinea'] = (
    df_limpio['Quantity'] * df_limpio['UnitPrice']
).round(2)

# Verificar la nueva columna
print("Registros:", len(df_limpio))
print("Valores faltantes:", df_limpio['ImporteLinea'].isna().sum())

print("\nEstadísticas de ImporteLinea:")
print(df_limpio['ImporteLinea'].describe().round(2))

# Mostrar ejemplos
display(
    df_limpio[
        ['InvoiceNo', 'Description', 'Quantity',
         'UnitPrice', 'ImporteLinea']
    ].head(10)
)

Registros: 541909
Valores faltantes: 0

Estadísticas de ImporteLinea:
count    541909.00
mean         17.99
std         378.81
min     -168469.60
25%           3.40
50%           9.75
75%          17.40
max      168469.60
Name: ImporteLinea, dtype: float64


,InvoiceNo,Description,Quantity,UnitPrice,ImporteLinea
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30
1,536365,WHITE METAL LANTERN,6,3.39,20.34
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34
5,536365,SET 7 BABUSHKA NESTING BOXES,2,7.65,15.30
6,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6,4.25,25.50
7,536366,HAND WARMER UNION JACK,6,1.85,11.10
8,536366,HAND WARMER RED POLKA DOT,6,1.85,11.10
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,1.69,54.08


_Por qué esta variable aporta a mi pregunta:_

Creé la variable ImporteLinea multiplicando la cantidad por el precio unitario de cada producto. Esta columna me permitirá calcular cuánto representa cada producto en las transacciones y comparar sus importes a lo largo del tiempo.

También será necesario diferenciar las ventas de las cancelaciones y ajustes contables, para que los resultados no se interpreten de manera incorrecta.



### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [28]:
# Actividad 3.2 — Normalización y codificación

# 1. Crear variables para el análisis temporal
df_limpio['Anio'] = df_limpio['InvoiceDate'].dt.year
df_limpio['Mes'] = df_limpio['InvoiceDate'].dt.month
df_limpio['AnioMes'] = (
    df_limpio['InvoiceDate'].dt.strftime('%Y-%m')
)

# 2. Identificar tipos de operaciones
cancelacion = df_limpio['InvoiceNo'].str.startswith('C', na=False)

ajuste = (
    df_limpio['Description']
    .astype('string')
    .str.contains('ADJUST', case=False, na=False)
    | (df_limpio['UnitPrice'] < 0)
)

df_limpio['TipoOperacion'] = np.select(
    [ajuste, cancelacion, df_limpio['Quantity'] < 0],
    ['AJUSTE', 'CANCELACION', 'CANTIDAD_NEGATIVA'],
    default='VENTA'
)

# 3. Codificar el tipo de operación
tipos_codificados = pd.get_dummies(
    df_limpio['TipoOperacion'],
    prefix='Tipo',
    dtype=int
)

# Eliminar columnas codificadas anteriores, si existen
columnas_anteriores = [
    c for c in df_limpio.columns
    if c.startswith('Tipo_')
]

df_limpio = df_limpio.drop(
    columns=columnas_anteriores
)

# Incorporar las columnas codificadas
df_limpio = pd.concat(
    [df_limpio, tipos_codificados],
    axis=1
)

# 4. Mostrar resultados
print("=== TIPOS DE OPERACIÓN ===")
print(df_limpio['TipoOperacion'].value_counts())

print("\n=== VARIABLES CREADAS ===")
print(tipos_codificados.columns.tolist())

display(
    df_limpio[
        ['InvoiceNo', 'InvoiceDate', 'AnioMes',
         'Quantity', 'UnitPrice', 'TipoOperacion']
    ].head(10)
)

print("\nDimensiones finales:", df_limpio.shape)

# Verificar que cada registro pertenezca a una sola categoría
columnas_tipo = [
    'Tipo_AJUSTE',
    'Tipo_CANCELACION',
    'Tipo_CANTIDAD_NEGATIVA',
    'Tipo_VENTA'
]

print(
    "Registros con clasificación incorrecta:",
    (df_limpio[columnas_tipo].sum(axis=1) != 1).sum()
)

print("Dimensiones:", df_limpio.shape)

print("Nombres de columnas duplicados:",
      df_limpio.columns.duplicated().sum())

=== TIPOS DE OPERACIÓN ===
TipoOperacion
VENTA                531261
CANCELACION            9288
CANTIDAD_NEGATIVA      1326
AJUSTE                   34
Name: count, dtype: int64

=== VARIABLES CREADAS ===
['Tipo_AJUSTE', 'Tipo_CANCELACION', 'Tipo_CANTIDAD_NEGATIVA', 'Tipo_VENTA']


,InvoiceNo,InvoiceDate,AnioMes,Quantity,UnitPrice,TipoOperacion
0,536365,2010-12-01 08:26:00,2010-12,6,2.55,VENTA
1,536365,2010-12-01 08:26:00,2010-12,6,3.39,VENTA
2,536365,2010-12-01 08:26:00,2010-12,8,2.75,VENTA
3,536365,2010-12-01 08:26:00,2010-12,6,3.39,VENTA
4,536365,2010-12-01 08:26:00,2010-12,6,3.39,VENTA
5,536365,2010-12-01 08:26:00,2010-12,2,7.65,VENTA
6,536365,2010-12-01 08:26:00,2010-12,6,4.25,VENTA
7,536366,2010-12-01 08:28:00,2010-12,6,1.85,VENTA
8,536366,2010-12-01 08:28:00,2010-12,6,1.85,VENTA
9,536367,2010-12-01 08:34:00,2010-12,32,1.69,VENTA



Dimensiones finales: (541909, 18)
Registros con clasificación incorrecta: 0
Dimensiones: (541909, 18)
Nombres de columnas duplicados: 0


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [29]:
# PARTE 4 — FUNCIÓN REUTILIZABLE DE LIMPIEZA
# Laboratorio 2 — Online Retail

def limpiar(ruta, semilla=SEMILLA):
    """Procesa Online Retail y devuelve (datos_preparados, informe)."""

    np.random.seed(semilla)

    # 1. Cargar nuevamente el archivo original
    d = pd.read_excel(ruta, engine='openpyxl')

    informe = {
        'filas_iniciales': len(d),
        'duplicados_detectados': int(d.duplicated().sum()),
        'customerid_faltantes': int(d['CustomerID'].isna().sum())
    }

    # 2. Centinelas
    # No se confirmaron centinelas para reemplazar por NaN.
    # Se conservan cantidades negativas y precios cero.

    # 3. Normalización de categorías
    def normalizar_categoria(serie):
        return (
            serie.astype('string')
            .str.strip()
            .str.upper()
            .str.normalize('NFKD')
            .str.encode('ascii', 'ignore')
            .str.decode('ascii')
        )

    equivalencias = {
        'EIRE': 'IRELAND'
    }

    d['Country_norm'] = (
        normalizar_categoria(d['Country'])
        .replace(equivalencias)
    )

    # 4. Duplicados
    # Se conservan porque no existe un ID único por línea.
    # Su posible efecto sobre ingresos queda documentado.

    # 5. Conversión y validación de tipos
    d['Quantity'] = pd.to_numeric(
        d['Quantity'], errors='coerce'
    )

    d['UnitPrice'] = pd.to_numeric(
        d['UnitPrice'], errors='coerce'
    )

    d['InvoiceDate'] = pd.to_datetime(
        d['InvoiceDate'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    d['CustomerID'] = (
        pd.to_numeric(d['CustomerID'], errors='coerce')
        .astype('Int64')
        .astype('string')
    )

    d['InvoiceNo'] = d['InvoiceNo'].astype('string')
    d['StockCode'] = d['StockCode'].astype('string')

    # 6. Valores atípicos
    # Se conservan las operaciones extremas.
    # Cancelaciones y ajustes se identifican posteriormente.

    # 7. Atributo derivado
    d['ImporteLinea'] = (
        d['Quantity'] * d['UnitPrice']
    ).round(2)

    # 8. Variables temporales
    d['Anio'] = d['InvoiceDate'].dt.year
    d['Mes'] = d['InvoiceDate'].dt.month
    d['AnioMes'] = d['InvoiceDate'].dt.strftime('%Y-%m')

    # 9. Clasificación de operaciones
    cancelacion = (
        d['InvoiceNo'].str.startswith('C', na=False)
    )

    ajuste = (
        d['Description']
        .astype('string')
        .str.contains('ADJUST', case=False, na=False)
        | (d['UnitPrice'] < 0)
    )

    d['TipoOperacion'] = np.select(
        [ajuste, cancelacion, d['Quantity'] < 0],
        ['AJUSTE', 'CANCELACION', 'CANTIDAD_NEGATIVA'],
        default='VENTA'
    )

    # 10. Codificar tipos de operación
    tipos_codificados = pd.get_dummies(
        d['TipoOperacion'],
        prefix='Tipo',
        dtype=int
    )

    d = pd.concat([d, tipos_codificados], axis=1)

    # 11. Informe final
    informe['filas_finales'] = len(d)
    informe['columnas_finales'] = len(d.columns)
    informe['filas_eliminadas'] = (
        informe['filas_iniciales'] - informe['filas_finales']
    )
    informe['tipos_operacion'] = (
        d['TipoOperacion'].value_counts().to_dict()
    )

    return d, informe


# Ejecutar el flujo completo
limpio, informe = limpiar(RUTA + ARCHIVO)

print("=== INFORME DE LIMPIEZA ===")

for clave, valor in informe.items():
    print(f"{clave}: {valor}")

print("\nDimensiones finales:", limpio.shape)
display(limpio.head())

=== INFORME DE LIMPIEZA ===
filas_iniciales: 541909
duplicados_detectados: 5268
customerid_faltantes: 135080
filas_finales: 541909
columnas_finales: 18
filas_eliminadas: 0
tipos_operacion: {'VENTA': 531261, 'CANCELACION': 9288, 'CANTIDAD_NEGATIVA': 1326, 'AJUSTE': 34}

Dimensiones finales: (541909, 18)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Country_norm,ImporteLinea,Anio,Mes,AnioMes,TipoOperacion,Tipo_AJUSTE,Tipo_CANCELACION,Tipo_CANTIDAD_NEGATIVA,Tipo_VENTA
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,UNITED KINGDOM,15.30,2010,12,2010-12,VENTA,0,0,0,1
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM,20.34,2010,12,2010-12,VENTA,0,0,0,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,UNITED KINGDOM,22.00,2010,12,2010-12,VENTA,0,0,0,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM,20.34,2010,12,2010-12,VENTA,0,0,0,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,UNITED KINGDOM,20.34,2010,12,2010-12,VENTA,0,0,0,1


In [30]:
# Comprobación obligatoria de reproducibilidad

a, informe_a = limpiar(RUTA + ARCHIVO)
b, informe_b = limpiar(RUTA + ARCHIVO)

print("¿Reproducible?:", a.equals(b))
print("¿Informes iguales?:", informe_a == informe_b)
print("Filas:", len(a))
print("Columnas:", len(a.columns))

assert a.equals(b), "Los resultados no son idénticos"

¿Reproducible?: True
¿Informes iguales?: True
Filas: 541909
Columnas: 18


In [31]:
# Guardar el conjunto preparado para la Semana 3

ARCHIVO_SALIDA = RUTA + 'S2_Online_Retail_Limpio.csv'

limpio.to_csv(
    ARCHIVO_SALIDA,
    sep=';',
    index=False,
    encoding='utf-8-sig',
    date_format='%Y-%m-%d %H:%M:%S'
)

print("Archivo guardado:", ARCHIVO_SALIDA)
print("Dimensiones:", limpio.shape)

Archivo guardado: S2_Online_Retail_Limpio.csv
Dimensiones: (541909, 18)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:**

https://github.com/fariasad/UCSC-Ciencia-de-Datos

---

## Antes de cerrar el laboratorio

- [x] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [x] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [x] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [x] Creé al menos un atributo derivado y expliqué para qué sirve
- [x] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [x] El conjunto limpio está guardado y versionado
- [x] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_